In [2]:

import sys, os
sys.path.append(os.path.abspath("..")) 
import json
import torch
from datasets import CNFDataset
from models import LightningModelCNF
from utils import args_cnf, gen_image_cnf, plotly_generate

**Read configuration files and arguments:**

In [3]:
# Arguments
parser = args_cnf()
args, unknown = parser.parse_known_args()

args.particle = "proton_contained"
args.metadata_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/metadata.pkl"
args.dataset_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/{}/{}/{}/{}.zip"
args.cnf_ind_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/gan_ind.pkl"
args.save_dir = "/pscratch/sd/b/botaoli/SFGD_VA/Results/cnf/"
args.checkpoint_path = "/pscratch/sd/b/botaoli/SFGD_VA/Results/cnf/test_vanilla/checkpoints"
args.checkpoint_name = args.particle

if args.particle == "muon" or args.particle == "proton_exiting":
    args.label_size = 10
elif args.particle == "proton_contained":
    args.label_size = 7
args.epochs = 50
args.log_every_n_steps = 2000
args.batch_size = 512
args.hidden = 256
args.num_workers = 64

**Load the pre-trained weights of the different generative-adversarial-network (GAN) models:**

In [4]:
# Dataset and generator models
test_set_p = CNFDataset(args, split="val")

checkpoint_path = "/".join((args.checkpoint_path, args.checkpoint_name, "train_loss", "epoch=73-step=103822.ckpt"))


model = LightningModelCNF.load_from_checkpoint(checkpoint_path, img_shape = (args.img_size, args.img_size, args.img_size), 
                                    label_size = args.label_size, 
                                    hidden_features = args.hidden, 
                                    num_blocks_in_MADE = args.num_blocks_in_MADE, 
                                    num_transformers = args.num_transformers, 
                                    lr = args.lr, 
                                    wd = args.weight_decay)

model.eval()

# move to cpu
model.to("cpu")


LightningModelCNF(
  (nflow): Flow(
    (_transform): CompositeTransform(
      (_transforms): ModuleList(
        (0-7): 8 x MaskedAffineAutoregressiveTransform(
          (autoregressive_net): MADE(
            (initial_layer): MaskedLinear(in_features=125, out_features=256, bias=True)
            (context_layer): Linear(in_features=7, out_features=256, bias=True)
            (activation): ReLU()
            (blocks): ModuleList(
              (0-1): 2 x MaskedFeedforwardBlock(
                (linear): MaskedLinear(in_features=256, out_features=256, bias=True)
                (activation): ReLU()
                (dropout): Dropout(p=0.0, inplace=False)
              )
            )
            (final_layer): MaskedLinear(in_features=256, out_features=250, bias=True)
          )
        )
        (8): RandomPermutation()
      )
    )
    (_distribution): StandardNormal()
    (_embedding_net): Identity()
  )
)

In [4]:
print(checkpoint_p['state_dict'].keys())

NameError: name 'checkpoint_p' is not defined

In [4]:
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['max'])
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['min'])

2658
3


In [4]:
for key, value in checkpoint_p['state_dict'].items():
    print(key, value.shape)

generator.bert.cls_token torch.Size([1, 1, 64])
generator.bert.embedding.input.weight torch.Size([64, 1])
generator.bert.embedding.input.bias torch.Size([64])
generator.bert.embedding.label.embedding.weight torch.Size([64, 10])
generator.bert.embedding.label.embedding.bias torch.Size([64])
generator.bert.embedding.position.vol_idx torch.Size([126])
generator.bert.embedding.position.embedding.weight torch.Size([126, 64])
generator.bert.embedding.noise.weight torch.Size([64, 512])
generator.bert.embedding.noise.bias torch.Size([64])
generator.bert.transformer_blocks.0.attention.linear_layers.0.weight torch.Size([64, 64])
generator.bert.transformer_blocks.0.attention.linear_layers.0.bias torch.Size([64])
generator.bert.transformer_blocks.0.attention.linear_layers.1.weight torch.Size([64, 64])
generator.bert.transformer_blocks.0.attention.linear_layers.1.bias torch.Size([64])
generator.bert.transformer_blocks.0.attention.linear_layers.2.weight torch.Size([64, 64])
generator.bert.transforme

In [9]:
print(checkpoint_p['state_dict']['critic.bert.embedding.input.weight'])

tensor([[ 0.5004],
        [-2.5577],
        [ 0.5532],
        [-9.0691],
        [ 2.0405],
        [ 1.9576],
        [ 0.0836],
        [ 0.5649],
        [-0.5833],
        [-0.5987],
        [ 1.0329],
        [ 5.5730],
        [ 1.9099],
        [ 4.0439],
        [ 0.7667],
        [ 1.3041],
        [ 0.2591],
        [ 1.5450],
        [ 6.0901],
        [ 2.3652],
        [ 3.0429],
        [-3.6318],
        [ 2.1024],
        [-0.3306],
        [ 0.4442],
        [ 5.3540],
        [ 5.7043],
        [-0.4855],
        [ 2.4442],
        [-0.2827],
        [-0.6130],
        [ 0.6491],
        [-1.9083],
        [ 1.0670],
        [-0.6086],
        [ 0.6371],
        [ 1.2413],
        [-1.5205],
        [ 5.3508],
        [ 0.5478],
        [-3.5981],
        [-1.0268],
        [ 0.4822],
        [ 0.7821],
        [-2.2264],
        [ 1.6432],
        [ 1.1424],
        [ 1.7548],
        [ 0.8087],
        [ 2.2181],
        [ 0.6237],
        [ 0.8127],
        [ 1.

**Run each GAN on some arbitrary input kinematics:**

In [62]:
'''
Proton GAN
'''
import numpy as np

# Set your kinematics here:
# ke = 30.3  # Initial kinetic energy
# ini_dir = [0.9999999999999999, 0.0, 0.0]  # Initial direction
# ini_pos = [-1.5, -4.2, 2.7]  # Initial 3D position (mm)

# get one event from the test set
event = test_set_p[32]
# convert torch tensor to numpy array
ke = float(event['ke'].numpy())
ini_pos = event['pos_ini'].numpy()
ini_dir = event['dir_ini'].numpy()
if args.particle == "proton_exiting" or args.particle == "muon":
    exit_pos = event['pos_exit'].numpy()
else:
    exit_pos = None
img = event['image'].numpy()

#print(ke, ini_pos, ini_dir, exit_pos, img)

if exit_pos is not None:
    params = np.array([ini_pos[0], ini_pos[1], ini_pos[2], ke, ini_dir[0], ini_dir[1], ini_dir[2], exit_pos[0], exit_pos[1], exit_pos[2]])
else:
    params = np.array([ini_pos[0], ini_pos[1], ini_pos[2], ke, ini_dir[0], ini_dir[1], ini_dir[2]])

params = np.array([0.1925,  0.2249, -0.0153, -0.1201, -0.4700, -0.4900, -0.7400])
labels = torch.tensor([params], dtype=torch.float32)

print(labels)




tensor([[ 0.1925,  0.2249, -0.0153, -0.1201, -0.4700, -0.4900, -0.7400]])


/tmp/ipykernel_375713/1137639086.py:14: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



In [63]:
with torch.no_grad():
    generated_p = model.sample(labels)

print(generated_p)

tensor([[[-0.9448, -1.0020, -0.9993, -1.0001, -0.9991, -1.0051, -1.0013,
          -1.0003, -0.9995, -1.0003, -1.0082, -1.0019, -1.0016, -1.0006,
          -1.0018, -0.9963, -0.9938, -0.9989, -0.9963, -1.0027, -0.9964,
          -1.0005, -1.0035, -0.9963, -1.0040, -0.9963, -0.9952, -0.9983,
          -0.9986, -0.9985, -0.9984, -0.9966, -1.0026, -0.9992, -0.9968,
          -0.9957, -1.0298, -0.9541, -1.0011, -0.9999, -1.0001, -1.0013,
          -1.0029, -1.0004, -0.9958, -0.9954, -0.9967, -1.0011, -0.9951,
          -0.9966, -0.9984, -0.9894, -0.9996, -1.0014, -1.0029, -0.9974,
          -0.9757, -0.9653, -1.0012, -0.9972, -0.9955, -0.6169, -0.2990,
          -0.9615, -1.0018, -1.0005, -1.0029, -1.0101, -1.0011, -0.9979,
          -1.0010, -1.0010, -0.9962, -0.9995, -1.0018, -0.9995, -0.9988,
          -0.9996, -0.9958, -1.0002, -0.9995, -0.9969, -1.0064, -0.9949,
          -0.9958, -0.9991, -0.9706, -0.9620, -1.0027, -0.9930, -1.0019,
          -1.0012, -1.0005, -0.9981, -1.0025, -1.00

tensor([[[  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000]],

        [[  3.0000,   3.0000,   3.0000,  39.9746,   3.0000],
         [  3.0000,   3.0000, 230.2248,   3.0000,   3.0000],
         [  3.0000,   3.0000, 100.7135,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000]],

        [[  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000, 228.2390,   3.0000,   3.0000],
         [  3.0000,   3.0000, 187.8737,   3.3160,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000,   3.0000,   3.0000,   3.0000,   3.0000]],

        [[  3.0000,   3.0000,   3.0000,   3.0000,   3.0000],
         [  3.0000

**Visualise the GAN-generated images:**

In [64]:
'''
Plot the generated images!
'''


#generated_p = generated_p.numpy()
# copy the image to a pure numpy array

print(generated_p.shape)
generated_p_1 = generated_p[0][0]

min_charge = 0
max_charge = test_set_p.metadata['statistics']['per_tree'][args.particle]['recon_charge']['max']

generated_p_1 = (generated_p_1 + 1) / 2
generated_p_1 *= (max_charge - min_charge)
generated_p_1 += min_charge

print(generated_p_1.shape)
# reshape the generated image to a 5x5x5 array
generated_p_1 = generated_p_1.reshape(5, 5, 5)



generated_plot = np.zeros((5, 5, 5))

for i in range(generated_plot.shape[0]):
    for j in range(generated_plot.shape[1]):
        for k in range(generated_plot.shape[2]):
            generated_plot[i, j, k] = float(generated_p_1[i, j, k])
print(generated_plot)
# check the type of the elements in the array
print(generated_plot.dtype)

# Max deposited energy in one voxel
max_energy = generated_plot.max()

#generated_plot[generated_plot > 150] = 0
generated_plot[generated_plot < 0] = 0
print(max_energy)




torch.Size([1, 1, 125])
torch.Size([125])
[[[ 7.33734512e+01 -2.59190083e+00  8.96867394e-01 -1.88213825e-01
    1.25000596e+00]
  [-6.79787779e+00 -1.76347482e+00 -4.44076896e-01  6.60015821e-01
   -3.62485886e-01]
  [-1.09577522e+01 -2.49811077e+00 -2.17348957e+00 -7.92462587e-01
   -2.44028425e+00]
  [ 4.85799217e+00  8.26968479e+00  1.47956979e+00  4.90575838e+00
   -3.60759020e+00]
  [ 4.74051666e+00 -6.79819465e-01 -4.70328617e+00  4.96968460e+00
   -5.26919508e+00]]

 [[ 4.88508368e+00  6.32971954e+00  2.25579333e+00  1.82288575e+00
    2.05324173e+00]
  [ 2.16834044e+00  4.55166912e+00 -3.44298220e+00  1.06662416e+00
    4.21682930e+00]
  [ 5.71350956e+00 -3.95667267e+01  6.09355736e+01 -1.47386634e+00
    1.59221292e-01]
  [-1.43695235e-01 -1.77583230e+00 -3.83509445e+00 -5.54026723e-01
    5.51689911e+00]
  [ 6.05619192e+00  4.34151316e+00 -1.45311213e+00  6.51072502e+00
    4.50057602e+00]]

 [[ 2.13047600e+00  1.40522690e+01  4.65306401e-01 -1.88752484e+00
   -3.90005016e+0

In [14]:
generated_plot = generated_plot.astype(np.float32)

In [65]:

print("- Proton image:")
plotly_generate(generated_plot, max_energy=max_energy)


- Proton image:


In [17]:
event = test_set_p[3]
true_img = np.zeros((125,))

for i in range(true_img.shape[0]):
    true_img[i] = float(event["image"][i])

true_img = true_img.reshape(5, 5, 5)

# get back the normalization
min_charge = 0
max_charge = test_set_p.metadata['statistics']['per_tree'][args.particle]['recon_charge']['max']
true_img = (true_img + 1) / 2
true_img *= (max_charge - min_charge)
true_img += min_charge

print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['std'])
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['mean'])

max_energy = true_img.max()

print(true_img)

print("- True image:")
plotly_generate(true_img, max_energy=max_energy)



303.3067761656923
197.1132585084357
[[[6.24434423e-01 3.85643702e-01 4.89664939e-01 9.88841943e-01
   6.58445799e-01]
  [9.42017971e-01 1.64223448e-01 5.52780583e-02 7.70990434e-01
   8.22162971e-01]
  [9.27364190e-01 7.57164235e-02 4.43002772e-01 3.12410520e-02
   9.00180852e-01]
  [3.49354316e-01 5.20336343e-01 3.83634867e-01 1.69582827e-01
   2.65458138e-01]
  [8.70462419e-01 5.62064531e-01 9.27277509e-01 4.30673944e-02
   9.24105940e-01]]

 [[3.88258251e-01 2.25221555e-01 1.55396866e-01 2.75788370e-01
   2.72722393e-01]
  [6.53030631e-01 2.92289467e-01 8.45073572e-01 5.47057080e-01
   3.80523750e-01]
  [4.50586901e-01 6.30227067e-01 4.61000113e-02 2.97822174e-01
   9.74632178e-01]
  [7.35434062e-01 2.45441606e-02 2.22230294e-01 3.38001828e-01
   2.42585807e-01]
  [6.29329529e-01 6.14863336e-02 7.89853662e-01 4.98551312e-02
   4.69097497e-01]]

 [[2.05748799e-01 3.23702696e-01 7.73012287e-01 2.32044355e-01
   7.63037045e-01]
  [3.62199238e-01 3.87748132e-01 6.42555977e-01 9.80030571

In [16]:
print(event["image"].numpy())

[ 197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  266.32901016  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  261.34530654  197.11325851  197.11325851
  197.11325851  263.76361099 1348.25107589  298.85057845  197.11325851
  197.11325851  197.11325851  286.39696517  197.11325851  197.11325851
  197.